# Scrapping niveau 2 — Titres des articles de 20 Minutes



In [1]:
import sys
!{sys.executable} -m pip install requests beautifulsoup4 lxml
# sqlite3 est inclus dans Python : rien a installer pour la base


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Imports et constantes

In [2]:
import re
import sqlite3
from datetime import datetime
from urllib.parse import urlparse

import requests
from bs4 import BeautifulSoup

URL = "https://www.20minutes.fr/"
HEADERS = {"User-Agent": "Mozilla/5.0"}
DB_PATH = "20minutes.db"

# Un vrai article a une URL du type /<categorie>/<id>-<date>-<slug>
ARTICLE_RE = re.compile(r"/\d{7}-\d{8}-")

## 3. Scrapping : récupérer titres et catégories

In [3]:
def scraper_articles(url=URL):
    """Renvoie une liste de dicts {categorie, titre} extraits de la page d'accueil."""
    reponse = requests.get(url, headers=HEADERS, timeout=20)
    reponse.raise_for_status()

    soup = BeautifulSoup(reponse.text, "lxml")
    articles = []
    vus = set()

    for balise in soup.find_all("article"):
        lien = balise.find("a", href=True)
        if not lien:
            continue

        href = lien["href"]
        # On ne garde que les vrais articles (pas les liens de rubrique)
        if not ARTICLE_RE.search(href):
            continue

        titre = lien.get_text(" ", strip=True)
        if not titre:
            continue

        # La categorie est le 1er segment du chemin de l'URL
        chemin = urlparse(href).path.lstrip("/").split("/")
        categorie = chemin[0] if chemin and chemin[0] else "inconnu"

        if titre in vus:  # eviter les doublons
            continue
        vus.add(titre)

        articles.append({"categorie": categorie, "titre": titre})

    return articles

## 4. Affichage dans la console

In [4]:
articles = scraper_articles()
print(f"{len(articles)} articles trouves\n")
for art in articles:
    print(f"[{art['categorie']:<12}] {art['titre']}")

134 articles trouves

[monde       ] Des dizaines de juifs ultra-orthodoxes ont pris d’assaut le domicile d’un m…
[monde       ] Une dizaine de pays de l’UE veulent restreindre l’entrée de touristes russe…
[justice     ] La perpétuité requise contre Martin Ney pour le meurtre de Jonathan
[arts-stars  ] Le faux scoop sur Xavier Dupont de Ligonnès a-t-il permis à M6 de cartonner…
[arts-stars  ] Arielle Dombasle ravie du concert d’Aya Nakamura au Stade de France
[arts-stars  ] La lettre de Jennifer Aniston à Matthew Perry retirée des enchères
[sport       ] « Tadej a regardé le Giro… » Pogacar aurait-il peur de Vingegaard ?
[sport       ] « Il regarde à chaque match… » Oui, Djokovic est à fond derrière Le Mans
[sport       ] Roland-Garros EN DIRECT : Pas de finale pour Halys et Herbert, Chwalinska v…
[high-tech   ] Fin de mission sur Mars, la sonde ne répond plus
[high-tech   ] L’astronaute Sophie Adenot voit « la marque de l’être humain sur la nature …
[high-tech   ] Des chercheurs trouv

## 5. Création de la base et de la table (étape 3 du sujet)

In [5]:
def creer_base(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS articles_20minutes (
            id        INTEGER PRIMARY KEY AUTOINCREMENT,
            date      TEXT NOT NULL,
            categorie TEXT NOT NULL,
            titre     TEXT NOT NULL
        )
        """
    )
    conn.commit()
    conn.close()
    print("Base et table pretes.")


creer_base()

Base et table pretes.


## 6. Enregistrement des données avec horodatage 

In [6]:
def enregistrer_articles(articles, db_path=DB_PATH):
    horodatage = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    conn = sqlite3.connect(db_path)
    conn.executemany(
        "INSERT INTO articles_20minutes (date, categorie, titre) VALUES (?, ?, ?)",
        [(horodatage, a["categorie"], a["titre"]) for a in articles],
    )
    conn.commit()
    nb = conn.total_changes
    conn.close()
    print(f"{nb} articles enregistres le {horodatage}.")


enregistrer_articles(articles)

134 articles enregistres le 2026-06-04 14:02:05.


## 7. Vérification : exemple « Roland Garros »

In [7]:
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    """
    SELECT date, categorie, titre
    FROM articles_20minutes
    WHERE titre LIKE '%Roland%'
      AND date LIKE '2026-06%'
    ORDER BY date DESC
    """
)
for ligne in cur.fetchall():
    print(ligne)
conn.close()

('2026-06-04 14:02:05', 'sport', 'Roland-Garros EN DIRECT\xa0: Pas de finale pour Halys et Herbert, Chwalinska v…')
('2026-06-04 14:02:05', 'sport', '07:25 Roland-Garros Andreeva et Schnaider, demi-finalistes russes très timides sur l’Ukraine L’absence de prise position claire des joueuses russes agace prodigieusement Marta Kostyuk, qui pourrait être amenée à les défier successivement en demie puis en finale')
('2026-06-04 14:02:05', 'sport', 'Roland-Garros EN DIRECT\xa0: Pas de finale pour Halys et Herbert, Chwalinska veut couper une nouvelle tête de série… Suivez les demies avec nous')


In [ ]:
# commande pour exporter BDD SQLite :  sqlite3 20minutes.db .dump > articles_20minutes.sql